# Stage B on Colab (무료 T4) — Qwen3-8B via Ollama

Stage A 계약(`query_iteration`)에 붙는 **Stage B 에이전트 루프**를 Colab 무료 T4에서
**Qwen3-8B**로 구동한다. LLM은 OpenAI 호환 엔드포인트로 접근하므로,
본선에서는 `base_url`만 프런티어 API로 바꾸면 코드 변경 없이 스위치된다.

> 런타임: **런타임 > 런타임 유형 변경 > T4 GPU** 선택 후 실행.

## 1. GPU 확인
T4(16GB)면 Qwen3-8B 4-bit가 여유롭게 올라간다. 14B는 KV 캐시가 빡빡하니 8B 권장.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ollama 설치 + 백그라운드 서버 기동
Ollama는 OpenAI 호환 `/v1/chat/completions`를 `localhost:11434`에 연다.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
# 백그라운드로 서버 기동
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
server = subprocess.Popen(['ollama','serve'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('ollama server up')

## 3. Qwen3-8B 내려받기
최초 1회 다운로드(수 GB). thinking/non-thinking 토글은 프롬프트의 `/no_think`로 제어한다.

In [ ]:
!ollama pull qwen3:8b

## 4. tool/JSON 출력 sanity check
Stage B는 구조화 JSON을 요구한다. 모델이 JSON만 뱉는지 먼저 확인.

In [ ]:
import requests
r = requests.post('http://localhost:11434/v1/chat/completions', json={
    'model':'qwen3:8b','temperature':0.2,
    'messages':[
        {'role':'system','content':'Respond with ONLY a JSON object.\n/no_think'},
        {'role':'user','content':'Return {"ok": true, "model": "qwen3"}'}]})
print(r.json()['choices'][0]['message']['content'])

## 5. FoS 리포 클론 + 설치
RDKit는 pip로 설치(합성가능성/valence 체크에 쓰임). 리포는 editable로.

In [ ]:
![ -d FoS ] || git clone https://github.com/leet1604/FoS.git
%cd FoS
!pip -q install rdkit pydantic requests
!pip -q install -e .

## 6. Stage B 실행 — Qwen3-8B 컨트롤러
fixture(EGFR/HER2) 위에서 Observe→Plan→Act→Assess 루프를 돈다.
- **Plan**은 `thinking=False`(토큰 절약), **Assess/Critic**만 thinking을 켜고 싶으면
  `ChatLLM(thinking=True)`로 별도 인스턴스를 만들어 loop에 주입하거나 아래처럼 한 개로 시작.
- 서버 응답이 느리거나 JSON이 깨지면 자동으로 HeuristicLLM로 폴백하니 루프는 멈추지 않는다.

In [ ]:
import shutil
from pathlib import Path
from stage_a.providers.fixture import SMILES
from stage_a.wiring import build_fixture_dependencies
from stage_b import ChatLLM, StageBConfig, run_stage_b

cache = Path('data/cache/contexts_colab'); shutil.rmtree(cache, ignore_errors=True)
deps = build_fixture_dependencies(str(cache))
config = StageBConfig(max_iterations=6, beam_k=3)

llm = ChatLLM(model='qwen3:8b', base_url='http://localhost:11434/v1',
              thinking=False, temperature=0.2)

result = run_stage_b(
    seed_smiles=SMILES['CHEMBL_M1'], on_target='CHEMBL203',
    dependencies=deps, llm=llm, config=config,
    auto_approve_top1=True, top_k_off_targets=3, verbose=True)

print('\n=== TRAJECTORY ===')
for s in result.trajectory:
    print(f'it{s.iteration} [{s.decision}] {s.rationale[:100]}')
print('\n=== FINAL BEAM (-> Stage C) ===')
for i,b in enumerate(result.final_beam):
    print(f'#{i} score={b.beam_score:+.3f} {b.position.canonical_smiles} S={b.position.selectivity_S}')

## 7. 실전 사용 팁

**Live ChEMBL로 돌리려면** `build_fixture_dependencies` 대신
`build_live_dependencies`를 쓰면 된다(ChEMBL API 호출 → 느리고 캐시 필요).

**thinking 모드 분리**: Critic 판단 품질이 아쉬우면 Assess만 thinking을 켠다.
`llm_backend.ChatLLM.assess`는 인스턴스의 `thinking`을 따르므로,
Plan용(`thinking=False`)과 Assess용(`thinking=True`) 인스턴스를 나눠 loop에
주입하도록 `run_stage_b`를 확장하면 토큰과 품질을 동시에 잡을 수 있다.

**본선 스위치**: 크레딧 받은 프런티어 API로 바꿀 때는
`ChatLLM(model='<api-model>', base_url='<api-base>/v1', api_key='<KEY>')`로
동일 인터페이스 그대로 교체.

**무료 T4 주의**: 세션은 idle 90분 / 최대 12시간에 끊긴다.
긴 루프는 `trajectory`를 파일로 자주 저장하고, 8B로 프로토타입한 뒤
최종 벤치 런만 API로 돌리는 걸 권장.